# 03 — Experiments and results

**Purpose:** Compare coordination scenarios, test attendance-shuffle nulls, benchmark against canonical random graph models, and verify compact `results/` files for submission.

**Regenerate:** `python scripts/run_analysis.py --task all`


In [6]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parent / "analysis"))
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
from notebook_paths import ensure_src_on_path

PROJECT_ROOT = ensure_src_on_path()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RESULTS_DIR = PROJECT_ROOT / "results"
DATA_DIR = PROJECT_ROOT / "data"


## Coordination scenarios (week 52 familiarity)

In [7]:
import pandas as pd
from soulcycle_network.analysis.io import load_master_table
from soulcycle_network.analysis.models import paired_scenario_comparison

master = load_master_table(OUTPUT_DIR)
final = master[(master["week"] == 52) & (master["network_type"] == "familiarity")]
paired_scenario_comparison(final, metric="edges", value_col="edges")


scenario,seed,baseline,high_coordination,no_coordination,baseline_minus_none,high_minus_none,baseline_percent_change,high_percent_change
0,6400,612,610,587,25,23,4.258944,3.918228
1,6401,567,541,615,-48,-74,-7.804878,-12.032520
2,6402,586,569,644,-58,-75,-9.006211,-11.645963
3,6403,574,568,682,-108,-114,-15.835777,-16.715543
4,6404,590,595,659,-69,-64,-10.470410,-9.711684
5,6405,624,628,720,-96,-92,-13.333333,-12.777778
6,6406,502,521,567,-65,-46,-11.463845,-8.112875
7,6407,561,574,591,-30,-17,-5.076142,-2.876481
8,6408,717,725,773,-56,-48,-7.244502,-6.209573
9,6409,527,529,565,-38,-36,-6.725664,-6.371681


## Null models and canonical comparisons

In [8]:
import pandas as pd

nulls = pd.read_csv(RESULTS_DIR / "null_models.csv")
comparisons = pd.read_csv(RESULTS_DIR / "model_comparisons.csv")
nulls.head(), comparisons.groupby("model", as_index=False)["degree_ks_distance_from_observed"].mean().round(4)


(          metric    observed       null      null_model
 0          nodes  522.000000  16.000000  global_shuffle
 1          edges  612.000000   8.000000  global_shuffle
 2        density    0.004501   0.066667  global_shuffle
 3    mean_degree    2.344828   1.000000  global_shuffle
 4  median_degree    1.000000   1.000000  global_shuffle,
                      model  degree_ks_distance_from_observed
 0           attractiveness                            0.0711
 1              erdos_renyi                            0.2291
 2                  fitness                            0.1602
 3                 observed                            0.0000
 4  preferential_attachment                            0.1199)

## Compact results bundle

In [9]:
import pandas as pd
from pathlib import Path

expected = [
    "calibration.csv",
    "longitudinal_metrics.csv",
    "null_models.csv",
    "model_comparisons.csv",
    "rider_nodes.csv",
    "familiarity_edges.csv",
    "social_edges.csv",
]
rows = []
for name in expected:
    p = RESULTS_DIR / name
    rows.append({
        "file": name,
        "exists": p.is_file(),
        "rows": len(pd.read_csv(p)) if p.is_file() else None,
    })
pd.DataFrame(rows)


,file,exists,rows
0,calibration.csv,True,10
1,longitudinal_metrics.csv,True,420
2,null_models.csv,True,26
3,model_comparisons.csv,True,41
4,rider_nodes.csv,True,10000
5,familiarity_edges.csv,True,612
6,social_edges.csv,True,39


## Takeaway and limitations

Coordination shifts edge counts relative to the no-coordination baseline; null models help separate structure from attendance constraints alone. ER/PA-style generators match some moments but not the full joint pattern of co-attendance.

**Limits:** synthetic riders, fixed studio system, no digital social graph, and a single boutique chain metaphor—not empirical SoulCycle data.


## Latent analog connections (week 52)

Counts on familiarity rows in `longitudinal_metrics.csv`—candidates a digital layer could surface, not friendships or recommendations.


In [10]:
latent_cols = [c for c in final.columns if c.startswith('latent_')]
final[latent_cols].iloc[0].to_frame('count') if latent_cols else 'Re-run run_analysis.py --task longitudinal'


,count
latent_pairs_one_encounter_from_familiarity,2179.0
latent_familiar_one_encounter_from_social,76.0
latent_repeated_co_not_social_dyads,2752.0
latent_riders_with_co_but_no_social_tie,8754.0
latent_cross_cluster_familiarity_ties,48.0
